# N07 — Summary: paper-ready tables & metric interpretation

**Purpose**: Load outputs of N03–N06, produce summary tables that mirror paper
Tables 5–13, and document interpretation + critical cautions for each metric.

This notebook does NOT edit the draft. It only produces artifacts for Phase 7
(paper value review).

**Outputs** (`outputs/N07/`): `table_*.csv`, `interpretation.md`

In [ ]:
import pickle, json
import numpy as np, pandas as pd
from pathlib import Path

N03=Path('../outputs/N03'); N04=Path('../outputs/N04')
N05=Path('../outputs/N05'); N06=Path('../outputs/N06')
OUT = Path('../outputs/N07'); OUT.mkdir(parents=True, exist_ok=True)

def load_bench(name):
    with open(N03/f'bench_{name}.pkl','rb') as f: return pickle.load(f)
benches = {n: load_bench(n) for n in ['GMSC','HC']}

## Table 5 + 6 — Prediction + Attribution fidelity (main)

In [ ]:
rows = []
for name, b in benches.items():
    for r in b['rows']:
        rows.append({'Dataset': name, **r})
df = pd.DataFrame(rows)[
    ['Dataset','surrogate','R2','Spearman','Agree','Top1','Top4',
     'AdvTop1','AdvTop4','AdvFull_R','AdvFull_J','coverage_surr']
].round(4)
df.to_csv(OUT/'table_5_6_main.csv', index=False)
print(df.to_string(index=False))

## Calibration summary

In [ ]:
with open(N04/'calibration_results.json') as f: cal = pd.DataFrame(json.load(f))
cal_show = cal[['dataset','surrogate','method','R2','AdvTop1','AdvTop4','AdvFull_R']].round(4)
cal_show.to_csv(OUT/'table_8_9_calibration.csv', index=False)
print(cal_show.to_string(index=False))

## Interventional fidelity summary (개입적 충실도)

In [ ]:
with open(N05/'interventional_results.json') as f: sic = pd.DataFrame(json.load(f))
print(sic.round(4).to_string(index=False))
sic.to_csv(OUT/'table_13_sic.csv', index=False)

## Cutoff + complexity

In [ ]:
with open(N06/'cutoff_results.json') as f: cut = pd.DataFrame(json.load(f))
with open(N06/'complexity_results.json') as f: cx = pd.DataFrame(json.load(f))
cut.round(4).to_csv(OUT/'table_10_cutoff.csv', index=False)
cx.round(4).to_csv(OUT/'table_11_complexity.csv', index=False)
print('cutoff:'); print(cut.round(4).to_string(index=False))
print('\ncomplexity:'); print(cx.round(4).to_string(index=False))

## Metric interpretation (written to `interpretation.md`)

- **R²** (teacher score ↔ surrogate score): how well the surrogate reproduces
  teacher scores. Essential but insufficient (Gosiewska 2020).
- **Spearman**: rank correlation. Less sensitive to scale than R².
- **Agree** (at percentile 10% cutoff): decision agreement on rejection.
  Robust to score scale.
- **Top-k**: identity overlap of top-k |contribution|. Sign-agnostic.
- **AdvTop-k**: identity overlap of top-k *adverse* contributions among rejects.
  Regulatory (ECOA 사유코드) relevance. k∈{1,4} canonical.
- **AdvFull_R / AdvFull_J**: full adverse-set Recall / Jaccard.
- **coverage_surr**: share of BB-used features present in surrogate. Warn when <1.0.
- **DA (Interventional fidelity)**: fraction of (sample, feature) pairs for which
  scorecard-suggested improvement moves teacher score in the right direction.
- **Spearman rho (Interventional fidelity)**: rank correlation of scorecard-predicted delta vs
  teacher's actual delta on the intervened pair. Calibration of magnitudes.
- **IR**: mean ratio teacher-delta / scorecard-delta. 1.0 = well calibrated.

In [ ]:
INTERP = '''# Metric interpretation

See notebook N07 for the full table. Summary of how each metric should be read in
the P5 context.

## Prediction fidelity
- R², Spearman, Agree. R² >= 0.9 conventional threshold.

## Attribution fidelity
- Top-k: general importance overlap.
- AdvTop-k: ECOA compliance proxy. k=1 = primary reason; k=4 = top 4 (FICO convention).
- AdvFull_R: recall of all BB adverse features.

## Interventional fidelity
- DC: directional correctness. >0.7 acceptable.
- Spearman rho: rank correlation of deltas.
- IR: magnitude calibration (1.0 ideal).

## Caveats
- coverage_surr < 1.0 ⇒ the surrogate is blind to some BB-used feature. Flag in paper.
- Random baseline provided in bench.info['random_baseline'] for AT-k floor.
'''
open(OUT/'interpretation.md','w', encoding='utf-8').write(INTERP)
print('Wrote', OUT/'interpretation.md')